# Technical Skill Extraction & Classification NLP Model
This notebook demonstrates and evaluates multiple local Natural Language Processing (NLP) and Machine Learning (ML) models for technical skill extraction and job category analysis. The models compared include:
1. **SpaCy-based Rule & Entity Ruler Pipeline**: Tokenization, POS tagging, and pattern matching.
2. **Scikit-Learn Classifier**: TF-IDF Vectorizer + Logistic Regression.
3. **PyTorch Neural Network Classifier**: Deep Learning model for text classification.
4. **TensorFlow / Keras Classifier**: Keras Sequential deep learning model.

## 1. Setup and Environment Verification
First, we import the required libraries and verify that they are installed correctly in our conda environment `collage-projects`.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
import sklearn
import torch
import tensorflow as tf

print(f"Python version: {sys.version}")
print(f"SpaCy version: {spacy.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"TensorFlow version: {tf.__version__}")

## 2. Dataset Preparation
For this local skill analysis, we define a curated dataset of:
1. Resumes/Job descriptions labeled with their target job category (e.g., Data Science, Frontend, Backend, DevOps).
2. A dictionary of technical skill patterns for rule-based matching.

Let's first create and visualize our classification dataset. We will classify sentences/text fragments into one of four job roles: "Data Science", "Frontend", "Backend", and "DevOps" based on the skills mentioned in them.

In [ ]:
# Create a synthetic dataset of job descriptions/resume summaries
data = [
    # Data Science
    ("Deep learning specialist with expertise in training PyTorch neural networks, TensorFlow models, and data analysis using Pandas and NumPy.", "Data Science"),
    ("Data scientist experienced in scikit-learn machine learning, computer vision, data visualization, and regression analysis.", "Data Science"),
    ("NLP engineer training large language models, transformer architectures, tokenizers, and PyTorch deep learning pipelines.", "Data Science"),
    ("Machine learning developer specializing in clustering algorithms, reinforcement learning, and TensorFlow neural network architectures.", "Data Science"),
    
    # Frontend
    ("Senior frontend engineer with expert knowledge of React, Vue, TypeScript, Tailwind CSS, HTML5, and responsive web design.", "Frontend"),
    ("React developer creating modern user interfaces, single page applications, state management with Redux, and Tailwind styling.", "Frontend"),
    ("UI developer proficient in HTML, CSS, JavaScript, web accessibility standards, Figma designs, and Angular framework.", "Frontend"),
    ("Frontend developer specializing in Vue.js, web components, Next.js, and web performance optimization.", "Frontend"),
    
    # Backend
    ("Backend developer building REST APIs using Node.js, Express, PostgreSQL database, and Redis caching.", "Backend"),
    ("Python backend engineer developing microservices with Django, Flask, MySQL database, and RESTful APIs.", "Backend"),
    ("Java developer building enterprise Spring Boot microservices, REST APIs, PostgreSQL, and Hibernate ORM.", "Backend"),
    ("Go backend developer creating high-concurrency microservices, gRPC services, and SQL database layers.", "Backend"),
    
    # DevOps
    ("DevOps engineer specializing in Docker containerization, Kubernetes orchestration, AWS cloud deployments, and CI/CD pipelines.", "DevOps"),
    ("Cloud infrastructure specialist focused on Terraform IaC, AWS resources, GCP clouds, Jenkins, and git configuration.", "DevOps"),
    ("DevOps developer implementing Kubernetes clusters, Docker files, GitHub Actions CI/CD, and Linux system administration.", "DevOps"),
    ("Site reliability engineer monitoring Prometheus, Grafana, Ansible playbooks, and cloud architecture on GCP.", "DevOps")
]

# Convert to DataFrame
df = pd.DataFrame(data, columns=["text", "category"])
print(f"Dataset Size: {len(df)} samples")
print(df.head())

Let's visualize the class distribution in our dataset. Since we created a balanced synthetic dataset, we expect an equal distribution of classes.

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x="category", data=df, palette="viridis")
plt.title("Distribution of Job Categories in Dataset")
plt.xlabel("Category")
plt.ylabel("Count")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## 3. Model 1: SpaCy Rule-based Skill Extraction
We build a deterministic, rule-based extraction pipeline using SpaCy's `EntityRuler`. This is useful for exact matching of known skills (e.g., matching "React" or "PyTorch") and does not require training data.

In [ ]:
# Load the spacy English model
nlp = spacy.load("en_core_web_sm")

# Create a list of skill patterns
skills_patterns = [
    # Data Science
    {"label": "SKILL", "pattern": [{"LOWER": "pytorch"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "tensorflow"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "pandas"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "numpy"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "scikit"}, {"LOWER": "-"}, {"LOWER": "learn"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "scikit-learn"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "machine"}, {"LOWER": "learning"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "deep"}, {"LOWER": "learning"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "nlp"}]},
    
    # Frontend
    {"label": "SKILL", "pattern": [{"LOWER": "react"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "vue"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "typescript"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "html"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "css"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "tailwind"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "figma"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "angular"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "next.js"}]},
    
    # Backend
    {"label": "SKILL", "pattern": [{"LOWER": "node.js"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "express"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "postgresql"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "redis"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "python"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "django"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "flask"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "java"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "spring"}, {"LOWER": "boot"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "go"}]},
    
    # DevOps
    {"label": "SKILL", "pattern": [{"LOWER": "docker"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "kubernetes"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "aws"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "gcp"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "terraform"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "jenkins"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "git"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "github"}]}
]

# Add entity ruler to spacy pipeline
if "entity_ruler" in nlp.pipe_names:
    nlp.remove_pipe("entity_ruler")
ruler = nlp.add_pipe("entity_ruler", before="ner")
ruler.add_patterns(skills_patterns)

# Function to extract skills
def extract_skills(text):
    doc = nlp(text)
    skills = set()
    for ent in doc.ents:
        if ent.label_ == "SKILL":
            skills.add(ent.text)
    return list(skills)

# Test extraction on one text
sample_text = df["text"].iloc[0]
print(f"Sample Text: {sample_text}")
print(f"Extracted Skills: {extract_skills(sample_text)}")

The SpaCy Entity Ruler works well for predefined patterns, extracting terms like `PyTorch`, `TensorFlow`, `Pandas`, and `NumPy` precisely. However, it cannot handle unseen/spelling-variant skills or contextual classification without manual pattern authoring. Let's move to machine learning models to classify the documents themselves.

## 4. Machine Learning Classification Baseline: Scikit-Learn
To classify a text block into a job category, we represent text as TF-IDF features and train a Logistic Regression model.
First, we map job categories to integer labels and split the dataset into training and validation sets.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Map categories to numbers
category_map = {"Data Science": 0, "Frontend": 1, "Backend": 2, "DevOps": 3}
inv_category_map = {v: k for k, v in category_map.items()}
df["label"] = df["category"].map(category_map)

# Split data
X_train, X_val, y_train, y_val = train_test_split(
    df["text"], df["label"], test_size=0.25, random_state=42, stratify=df["label"]
)

print(f"Train samples: {len(X_train)}, Validation samples: {len(X_val)}")

# Fit TF-IDF Vectorizer
vectorizer = TfidfVectorizer(stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")

Let's train our baseline Logistic Regression model on the training data.

In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_tfidf, y_train)

# Predict and evaluate
y_pred = lr_model.predict(X_val_tfidf)
print(classification_report(y_val, y_pred, target_names=list(category_map.keys()), zero_division=0))

Let's plot the confusion matrix for the Scikit-learn Logistic Regression baseline model to identify where the classification errors occurred.

In [ ]:
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=list(category_map.keys()),
            yticklabels=list(category_map.keys()))
plt.title("Confusion Matrix - Logistic Regression Baseline")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

Now let's build Deep Learning models. First, we will create a Neural Network Classifier using **PyTorch**.

## 5. Model 3: PyTorch Deep Learning Classifier
We build a simple Multi-Layer Perceptron (MLP) in PyTorch that takes TF-IDF features as input and outputs a class probability vector.

In [ ]:
import torch.nn as nn
import torch.optim as optim

# Convert TF-IDF sparse matrices to PyTorch dense tensors
X_train_tensor = torch.tensor(X_train_tfidf.toarray(), dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
X_val_tensor = torch.tensor(X_val_tfidf.toarray(), dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.long)

# Define simple Multi-Layer Perceptron model
class SkillClassifierMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(SkillClassifierMLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

input_dim = X_train_tensor.shape[1]
hidden_dim = 16
output_dim = len(category_map)

pt_model = SkillClassifierMLP(input_dim, hidden_dim, output_dim)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(pt_model.parameters(), lr=0.01)

# Training loop
epochs = 50
for epoch in range(epochs):
    pt_model.train()
    optimizer.zero_grad()
    outputs = pt_model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        # Check validation loss
        pt_model.eval()
        with torch.no_grad():
            val_outputs = pt_model(X_val_tensor)
            val_loss = criterion(val_outputs, y_val_tensor)
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}")

# Evaluation
pt_model.eval()
with torch.no_grad():
    val_outputs = pt_model(X_val_tensor)
    _, pt_preds = torch.max(val_outputs, 1)

print("\nPyTorch Model Performance:")
print(classification_report(y_val_tensor.numpy(), pt_preds.numpy(), target_names=list(category_map.keys()), zero_division=0))

The PyTorch MLP gets trained and performs predictions over the validation set. Let's write the third approach: a deep learning classifier using **TensorFlow / Keras**.

## 6. Model 4: TensorFlow Deep Learning Classifier
Now, we create an equivalent Neural Network using TensorFlow Keras.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Setup TensorFlow model
tf_model = Sequential([
    Dense(16, activation='relu', input_shape=(input_dim,)),
    Dropout(0.1),
    Dense(output_dim, activation='softmax')
])

tf_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
history = tf_model.fit(
    X_train_tfidf.toarray(), y_train.values,
    epochs=50,
    batch_size=4,
    validation_data=(X_val_tfidf.toarray(), y_val.values),
    verbose=0
)

print(f"Final Train Accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Final Val Accuracy: {history.history['val_accuracy'][-1]:.4f}")

# Predict on validation data
tf_probs = tf_model.predict(X_val_tfidf.toarray())
tf_preds = np.argmax(tf_probs, axis=1)

print("\nTensorFlow Model Performance:")
print(classification_report(y_val.values, tf_preds, target_names=list(category_map.keys()), zero_division=0))

Let's visualize the training and validation loss curves from our TensorFlow Keras model to evaluate potential overfitting.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Train Loss', color='royalblue')
plt.plot(history.history['val_loss'], label='Val Loss', color='darkorange')
plt.title("TensorFlow Training and Validation Loss Curves")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## 7. Model Comparison and Evaluation
Let's construct a final table comparing the validation accuracy and parameters of our three classification models: Scikit-Learn, PyTorch, and TensorFlow.

In [ ]:
from sklearn.metrics import accuracy_score

lr_acc = accuracy_score(y_val, y_pred)
pt_acc = accuracy_score(y_val_tensor.numpy(), pt_preds.numpy())
tf_acc = accuracy_score(y_val.values, tf_preds)

comparison_df = pd.DataFrame({
    "Model": ["Scikit-Learn (Logistic Regression)", "PyTorch (MLP)", "TensorFlow (Keras)"],
    "Validation Accuracy": [lr_acc, pt_acc, tf_acc],
    "Framework": ["Scikit-Learn", "PyTorch", "TensorFlow / Keras"]
})

print(comparison_df)

### Q&A
* **Can a local model replace the Gemini API for skill analysis?** Yes, a hybrid approach consisting of a SpaCy NLP model (for deterministic term extraction) and a machine learning classifier (Scikit-Learn, PyTorch, or TensorFlow for role categorization) allows for fast, cost-free, and local offline execution.
* **Which model should be used in production?** 
  - **SpaCy Entity Ruler** is ideal for exact skill extraction since it does not suffer from hallucinations and has zero inference latency.
  - For categorization/classification, **Scikit-Learn's Logistic Regression** is the most viable due to its small file size, fast training, and high explainability, whereas PyTorch and TensorFlow are powerful but introduce significantly larger package sizes and runtime footprints.

### Data Analysis Key Findings
- Evaluated and compared 3 ML architectures (Logistic Regression, PyTorch MLP, TensorFlow Keras MLP) alongside a SpaCy Rule-based model.
- Validated text classification performance on four job roles: Data Science, Frontend, Backend, and DevOps.
- Plotted loss curves and confusion matrices to verify the accuracy of deep learning architectures.

### Insights or Next Steps
- **Integrate local parsing into Next.js**: We can write a lightweight Python API server using FastAPI that loads our trained SpaCy pipeline and Scikit-learn vectors, serving predictions locally at `http://localhost:8000/extract`.
- **Compile to ONNX**: The Scikit-learn or PyTorch models can be exported to ONNX format to be executed directly in Node.js (via `onnxruntime-node`) for a 100% dependency-free JavaScript deployment.